# PCA Reproduction from the Merged 310-Site Chemical Block

This notebook extracts the 16-chemical block from the merged 310-site dataset, keeps each chemical on its default stored scale, applies `log10(x + 1)`, runs PCA on the correlation matrix, computes the first five original loading columns, applies quartimax rotation to those five loading columns, and displays the original PCA scores from the fit stage.

In [137]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

N_COMPONENTS = 5
TO_UGG = {
    'ug/g': 1.0,
    'µg/g': 1.0,
    'μg/g': 1.0,
    'mg/g': 1000.0,
    'ng/g': 0.001,
}


def find_project_root(start: Path | None = None) -> Path:
    current = (Path.cwd() if start is None else Path(start)).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'setup.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root from the current working directory.')


def load_merged_chemical_block(workbook_path: Path) -> pd.DataFrame:
    multiheader_df = pd.read_excel(workbook_path, sheet_name='merged_data', header=[0, 1])
    if isinstance(multiheader_df.columns, pd.MultiIndex):
        top_level = set(multiheader_df.columns.get_level_values(0))
        if {'sample_info', 'chemical'}.issubset(top_level):
            station_df = multiheader_df.loc[:, [('sample_info', 'StationID')]].copy()
            station_df.columns = ['StationID']
            chemical_df = multiheader_df['chemical'].copy()
            chemical_df.columns = chemical_df.columns.astype(str)
            return pd.concat([station_df, chemical_df], axis=1)

    raw = pd.read_excel(workbook_path, sheet_name='merged_data', header=None)
    variable_names = ['StationID', *raw.iloc[2, 1:].tolist()]
    data = raw.iloc[4:].reset_index(drop=True).copy()
    data.columns = variable_names
    return data


def load_unit_map(merged_path: Path, fallback_path: Path) -> pd.DataFrame:
    merged_book = pd.ExcelFile(merged_path)
    if len(merged_book.sheet_names) >= 3:
        third_sheet = pd.read_excel(merged_path, sheet_name=merged_book.sheet_names[2])
        lowered = {str(column).strip().lower(): column for column in third_sheet.columns}
        if 'chemical' in lowered and ('target_unit' in lowered or 'unit' in lowered):
            unit_col = lowered.get('target_unit', lowered['unit'])
            return third_sheet[[lowered['chemical'], unit_col]].rename(
                columns={lowered['chemical']: 'Chemical', unit_col: 'unit'}
            )

    chemical_info = pd.read_excel(fallback_path, sheet_name='chemical_info')
    return chemical_info[['Chemical', 'target_unit']].rename(columns={'target_unit': 'unit'})


def orthomax_rotation(loadings: np.ndarray, gamma: float, max_iter: int = 1000, tol: float = 1e-7) -> tuple[np.ndarray, np.ndarray]:
    if loadings.shape[1] <= 1:
        return loadings.copy(), np.eye(loadings.shape[1])

    n_rows, n_cols = loadings.shape
    rotation = np.eye(n_cols)
    previous_objective = 0.0

    for _ in range(max_iter):
        rotated = loadings @ rotation
        transformed = loadings.T @ (
            rotated ** 3 - (gamma / n_rows) * rotated @ np.diag(np.diag(rotated.T @ rotated))
        )
        left, singular_values, right_t = np.linalg.svd(transformed)
        rotation = left @ right_t
        objective = singular_values.sum()
        if objective - previous_objective <= tol:
            break
        previous_objective = objective

    return loadings @ rotation, rotation


def validate_chemical_block(dataframe: pd.DataFrame, chemical_columns: list[str]) -> pd.DataFrame:
    validated_df = dataframe[['StationID', *chemical_columns]].copy()
    validated_df['StationID'] = validated_df['StationID'].astype('string').str.strip()

    for chemical in chemical_columns:
        validated_df[chemical] = pd.to_numeric(validated_df[chemical], errors='coerce')

    missing_counts = validated_df[chemical_columns].isna().sum()
    missing_counts = missing_counts[missing_counts > 0]
    if not missing_counts.empty:
        raise ValueError(f'Chemical block contains missing values: {missing_counts.to_dict()}')

    minimum_value = validated_df[chemical_columns].min().min()
    if minimum_value < 0:
        raise ValueError(
            'Chemical block contains negative values, so log2(x + 1) is invalid. '
            f'Minimum value: {minimum_value}'
        )

    return validated_df


def convert_chemical_block_to_ugg(
    dataframe: pd.DataFrame, unit_lookup_df: pd.DataFrame, chemical_columns: list[str]
 ) -> tuple[pd.DataFrame, pd.DataFrame]:
    conversion_df = unit_lookup_df[['Chemical', 'unit']].copy()
    conversion_df['factor_to_ugg'] = conversion_df['unit'].map(TO_UGG)

    missing_units = conversion_df.loc[conversion_df['factor_to_ugg'].isna(), ['Chemical', 'unit']]
    if not missing_units.empty:
        raise ValueError(
            'Unsupported chemical units for ug/g conversion: '
            + str(missing_units.to_dict(orient='records'))
        )

    converted_df = dataframe.copy()
    factor_map = dict(zip(conversion_df['Chemical'], conversion_df['factor_to_ugg']))
    for chemical in chemical_columns:
        converted_df[chemical] = converted_df[chemical] * factor_map[chemical]

    conversion_df['unit_used_for_pca'] = 'ug/g'
    return converted_df, conversion_df[['Chemical', 'unit', 'factor_to_ugg', 'unit_used_for_pca']]


def run_rotated_pca_pipeline(
    dataframe: pd.DataFrame, chemical_columns: list[str], rotation_name: str, gamma: float
 ) -> dict[str, pd.DataFrame]:
    log10_df = np.log10(dataframe[chemical_columns] + 1.0)
    standardized_df = (log10_df - log10_df.mean(axis=0)) / log10_df.std(axis=0, ddof=1)
    correlation_matrix = standardized_df.corr()

    eigenvalues, eigenvectors = np.linalg.eigh(correlation_matrix.to_numpy(dtype=float))
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    component_names = [f'PC{i}' for i in range(1, N_COMPONENTS + 1)]
    selected_eigenvalues = eigenvalues[:N_COMPONENTS]
    selected_eigenvectors = eigenvectors[:, :N_COMPONENTS]

    original_loadings = selected_eigenvectors * np.sqrt(selected_eigenvalues)
    rotated_loadings, _ = orthomax_rotation(original_loadings, gamma=gamma)
    original_scores = standardized_df.to_numpy(dtype=float) @ selected_eigenvectors

    original_loadings_df = pd.DataFrame(
        original_loadings, index=chemical_columns, columns=component_names
    ).round(4)
    rotated_loadings_df = pd.DataFrame(
        rotated_loadings, index=chemical_columns, columns=component_names
    ).round(4)
    pc_scores_df = pd.DataFrame(original_scores, columns=component_names).round(4)
    pc_scores_df.insert(0, 'StationID', dataframe['StationID'].to_numpy())

    return {
        'rotation_name': rotation_name,
        'original_loadings_df': original_loadings_df,
        'rotated_loadings_df': rotated_loadings_df,
        'pc_scores_df': pc_scores_df,
    }


def run_pca_pipeline(dataframe: pd.DataFrame, chemical_columns: list[str]) -> dict[str, pd.DataFrame]:
    quartimax_results = run_rotated_pca_pipeline(
        dataframe, chemical_columns, rotation_name='quartimax', gamma=0.0
    )
    return {
        'original_loadings_df': quartimax_results['original_loadings_df'],
        'quartimax_loadings_df': quartimax_results['rotated_loadings_df'],
        'pc_scores_df': quartimax_results['pc_scores_df'],
    }


project_root = find_project_root()
final_cleaning_dir = project_root / 'CodeBase' / 'MetaData' / 'FinalCleaningWork'
merged_workbook_path = final_cleaning_dir / 'merged_310_sites_taxa_chemical_environmental_data.xlsx'
chemical_workbook_path = final_cleaning_dir / 'finalized_datasets' / '322_sites_chemical_data.xlsx'

merged_df = load_merged_chemical_block(merged_workbook_path)
unit_df = load_unit_map(merged_workbook_path, chemical_workbook_path)
chemical_columns = unit_df['Chemical'].tolist()

missing_columns = [column for column in ['StationID', *chemical_columns] if column not in merged_df.columns]
if missing_columns:
    raise ValueError(f'Missing required merged-data columns: {missing_columns}')

chemical_default_scale_df = validate_chemical_block(merged_df, chemical_columns)
distinct_input_units = sorted(unit_df['unit'].dropna().astype(str).unique().tolist())

## PCA using the default stored unit array

This section confirms the unit array carried by the merged chemical block and runs the PCA directly on those stored values after `log10(x + 1)`.

In [138]:
print('Unit array carried by the merged chemical block:')
display(unit_df.reset_index(drop=True))
print(f'Distinct input units: {distinct_input_units}')

default_unit_results = run_pca_pipeline(chemical_default_scale_df, chemical_columns)

print('\nOriginal PCA loadings (first 5 PCs)')
display(default_unit_results['original_loadings_df'])

print('\nQuartimax-rotated PCA loadings (first 5 PCs)')
display(default_unit_results['quartimax_loadings_df'])

print('\nOriginal PCA scores from the fit stage')
display(default_unit_results['pc_scores_df'])

Unit array carried by the merged chemical block:


,Chemical,unit
0,Co,ug/g
1,Al,ug/g
2,Ni,ug/g
3,Mn,ug/g
4,Fe,ug/g
5,Cr,ug/g
6,Cu,ug/g
7,Hg,ng/g
8,Pb,ug/g
9,Zn,ug/g


Distinct input units: ['ng/g', 'ug/g']

Original PCA loadings (first 5 PCs)


,PC1,PC2,PC3,PC4,PC5
Co,0.8230,0.1877,-0.3408,-0.1944,0.0179
Al,0.6832,0.4637,-0.3843,-0.1928,0.0348
Ni,0.8030,-0.1462,0.0996,0.2238,0.3657
Mn,0.6936,0.5171,0.1993,0.2591,0.0811
Fe,0.6307,0.1856,-0.1375,0.0429,0.0831
Cr,0.9372,-0.0343,0.0616,-0.0202,0.1982
Cu,0.8795,-0.0585,0.0360,-0.0928,0.2146
Hg,0.5833,-0.1035,-0.4448,-0.3633,-0.3813
Pb,0.4951,0.2299,0.7608,0.0261,-0.1165
Zn,0.8147,-0.3172,-0.0408,-0.0931,0.1792



Quartimax-rotated PCA loadings (first 5 PCs)


,PC1,PC2,PC3,PC4,PC5
Co,0.7211,0.4271,-0.0117,-0.0363,-0.4035
Al,0.5226,0.6624,-0.0110,-0.0884,-0.3848
Ni,0.8800,0.0912,-0.0333,0.0410,0.2726
Mn,0.5369,0.6275,0.3262,0.1311,0.2370
Fe,0.5614,0.3667,0.0180,0.0516,-0.0856
Cr,0.9327,0.1848,0.1332,-0.0357,-0.0002
Cu,0.8880,0.1474,0.1041,-0.1005,-0.0413
Hg,0.4743,0.0683,0.0233,0.1028,-0.7651
Pb,0.3771,0.1173,0.7959,-0.0004,0.3188
Zn,0.8861,-0.0891,-0.0169,-0.0426,-0.1075



Original PCA scores from the fit stage


,StationID,PC1,PC2,PC3,PC4,PC5
0,003ABC,-1.0317,1.3996,1.1579,-0.9134,-1.8061
1,004ABC,-3.6343,-0.4974,-1.6005,-0.6813,-0.0455
2,005ABC,-3.0874,-0.1495,1.1548,-1.5793,-1.6389
3,007ABC,3.3931,1.6862,0.1916,-1.5063,-1.7204
4,008A,-2.1891,0.4878,1.0950,-1.6538,-2.0426
...,...,...,...,...,...,...
305,S99,-2.5671,-0.8230,0.3854,-0.3841,0.6797
306,UBC1,-0.4744,0.5121,-1.0347,2.4312,-1.6386
307,UCC1,-0.7001,0.7921,-0.6825,1.4729,-0.9113
308,UCE1,-0.6913,1.4563,-0.0301,0.8353,-0.4395


## PCA after converting every chemical to ug/g

This section converts the same 16 chemicals into a uniform `ug/g` unit array first, then applies the same `log10(x + 1)` and PCA workflow.

In [139]:
chemical_ugg_df, ugg_conversion_df = convert_chemical_block_to_ugg(
    chemical_default_scale_df, unit_df, chemical_columns
)

print('Unit conversion applied before PCA:')
display(ugg_conversion_df.reset_index(drop=True))

ugg_unit_results = run_pca_pipeline(chemical_ugg_df, chemical_columns)

print('\nOriginal PCA loadings (first 5 PCs)')
display(ugg_unit_results['original_loadings_df'])

print('\nQuartimax-rotated PCA loadings (first 5 PCs)')
display(ugg_unit_results['quartimax_loadings_df'])

print('\nOriginal PCA scores from the fit stage')
display(ugg_unit_results['pc_scores_df'])

Unit conversion applied before PCA:


,Chemical,unit,factor_to_ugg,unit_used_for_pca
0,Co,ug/g,1.000,ug/g
1,Al,ug/g,1.000,ug/g
2,Ni,ug/g,1.000,ug/g
3,Mn,ug/g,1.000,ug/g
4,Fe,ug/g,1.000,ug/g
5,Cr,ug/g,1.000,ug/g
6,Cu,ug/g,1.000,ug/g
7,Hg,ng/g,0.001,ug/g
8,Pb,ug/g,1.000,ug/g
9,Zn,ug/g,1.000,ug/g



Original PCA loadings (first 5 PCs)


,PC1,PC2,PC3,PC4,PC5
Co,0.8191,0.3459,-0.2642,-0.0042,0.0801
Al,0.6900,0.5332,-0.2177,0.1937,0.0783
Ni,0.7921,-0.1097,0.0664,-0.1204,-0.4558
Mn,0.7213,0.1560,0.3095,0.4500,-0.1248
Fe,0.6417,0.1032,-0.1126,0.1831,-0.0610
Cr,0.9368,0.0117,0.0258,-0.1352,-0.1555
Cu,0.8784,0.0366,0.0033,-0.2157,-0.1620
Hg,0.5020,0.3266,-0.4815,-0.2187,0.4391
Pb,0.5181,-0.1303,0.7584,0.0519,0.1278
Zn,0.7870,-0.0262,-0.1038,-0.3767,-0.1716



Quartimax-rotated PCA loadings (first 5 PCs)


,PC1,PC2,PC3,PC4,PC5
Co,0.7307,-0.0005,0.0246,0.2822,0.5026
Al,0.5621,0.1118,0.0209,0.5101,0.5122
Ni,0.8946,-0.0576,-0.0681,0.0726,-0.2294
Mn,0.5792,-0.0677,0.2915,0.6467,-0.1232
Fe,0.5596,-0.1424,0.0115,0.3354,0.1621
Cr,0.9404,-0.0796,0.1233,0.1024,0.0666
Cu,0.9103,-0.0218,0.0903,0.0294,0.0864
Hg,0.3886,-0.0334,0.0316,-0.0378,0.8226
Pb,0.4153,-0.0180,0.7537,0.1416,-0.3445
Zn,0.8722,-0.0215,-0.0099,-0.1634,0.1192



Original PCA scores from the fit stage


,StationID,PC1,PC2,PC3,PC4,PC5
0,003ABC,-0.9420,0.4268,1.5492,1.0301,1.9349
1,004ABC,-3.4659,-0.0834,-1.6081,-0.2754,-0.0296
2,005ABC,-2.9909,-0.4385,1.0710,-0.5193,2.0940
3,007ABC,3.5053,1.9372,0.5261,-0.0271,2.3087
4,008A,-1.9699,-0.1024,1.1764,-0.3319,2.3487
...,...,...,...,...,...,...
305,S99,-2.5076,-0.6398,0.1162,-0.7184,-0.4045
306,UBC1,-0.6351,-0.6503,-0.7196,1.6332,0.1219
307,UCC1,-0.8381,0.1675,-0.1779,1.1082,-0.2192
308,UCE1,-0.5970,0.4528,0.5245,1.1038,-0.1562


## PCA using varimax rotation on the default stored unit array

This section keeps the mixed default unit array from the merged chemical block and replaces quartimax with varimax rotation before displaying the same three PCA outputs.

In [140]:
varimax_default_unit_results = run_rotated_pca_pipeline(
    chemical_default_scale_df, chemical_columns, rotation_name='varimax', gamma=1.0
)

print('Original PCA loadings (first 5 PCs)')
display(varimax_default_unit_results['original_loadings_df'])

print('\nVarimax-rotated PCA loadings (first 5 PCs)')
display(varimax_default_unit_results['rotated_loadings_df'])

print('\nOriginal PCA scores from the fit stage')
display(varimax_default_unit_results['pc_scores_df'])

Original PCA loadings (first 5 PCs)


,PC1,PC2,PC3,PC4,PC5
Co,0.8230,0.1877,-0.3408,-0.1944,0.0179
Al,0.6832,0.4637,-0.3843,-0.1928,0.0348
Ni,0.8030,-0.1462,0.0996,0.2238,0.3657
Mn,0.6936,0.5171,0.1993,0.2591,0.0811
Fe,0.6307,0.1856,-0.1375,0.0429,0.0831
Cr,0.9372,-0.0343,0.0616,-0.0202,0.1982
Cu,0.8795,-0.0585,0.0360,-0.0928,0.2146
Hg,0.5833,-0.1035,-0.4448,-0.3633,-0.3813
Pb,0.4951,0.2299,0.7608,0.0261,-0.1165
Zn,0.8147,-0.3172,-0.0408,-0.0931,0.1792



Varimax-rotated PCA loadings (first 5 PCs)


,PC1,PC2,PC3,PC4,PC5
Co,0.4582,0.5992,0.0800,-0.0046,-0.5397
Al,0.2056,0.7646,0.0517,-0.0582,-0.4847
Ni,0.8326,0.3723,0.1223,0.0630,0.0956
Mn,0.2838,0.7524,0.4213,0.1554,0.1165
Fe,0.3810,0.5104,0.1020,0.0743,-0.1958
Cr,0.7780,0.4517,0.2794,-0.0064,-0.1885
Cu,0.7471,0.4032,0.2417,-0.0727,-0.2201
Hg,0.2727,0.1552,0.0515,0.1261,-0.8421
Pb,0.2411,0.2020,0.8634,0.0125,0.2156
Zn,0.8263,0.1793,0.1169,-0.0189,-0.2789



Original PCA scores from the fit stage


,StationID,PC1,PC2,PC3,PC4,PC5
0,003ABC,-1.0317,1.3996,1.1579,-0.9134,-1.8061
1,004ABC,-3.6343,-0.4974,-1.6005,-0.6813,-0.0455
2,005ABC,-3.0874,-0.1495,1.1548,-1.5793,-1.6389
3,007ABC,3.3931,1.6862,0.1916,-1.5063,-1.7204
4,008A,-2.1891,0.4878,1.0950,-1.6538,-2.0426
...,...,...,...,...,...,...
305,S99,-2.5671,-0.8230,0.3854,-0.3841,0.6797
306,UBC1,-0.4744,0.5121,-1.0347,2.4312,-1.6386
307,UCC1,-0.7001,0.7921,-0.6825,1.4729,-0.9113
308,UCE1,-0.6913,1.4563,-0.0301,0.8353,-0.4395


## PCA using varimax rotation after converting every chemical to ug/g

This section first converts the same 16 chemicals to a uniform `ug/g` unit array and then applies varimax rotation before displaying the same three PCA outputs.

In [141]:
varimax_ugg_unit_results = run_rotated_pca_pipeline(
    chemical_ugg_df, chemical_columns, rotation_name='varimax', gamma=1.0
)

print('Original PCA loadings (first 5 PCs)')
display(varimax_ugg_unit_results['original_loadings_df'])

print('\nVarimax-rotated PCA loadings (first 5 PCs)')
display(varimax_ugg_unit_results['rotated_loadings_df'])

print('\nOriginal PCA scores from the fit stage')
display(varimax_ugg_unit_results['pc_scores_df'])

Original PCA loadings (first 5 PCs)


,PC1,PC2,PC3,PC4,PC5
Co,0.8191,0.3459,-0.2642,-0.0042,0.0801
Al,0.6900,0.5332,-0.2177,0.1937,0.0783
Ni,0.7921,-0.1097,0.0664,-0.1204,-0.4558
Mn,0.7213,0.1560,0.3095,0.4500,-0.1248
Fe,0.6417,0.1032,-0.1126,0.1831,-0.0610
Cr,0.9368,0.0117,0.0258,-0.1352,-0.1555
Cu,0.8784,0.0366,0.0033,-0.2157,-0.1620
Hg,0.5020,0.3266,-0.4815,-0.2187,0.4391
Pb,0.5181,-0.1303,0.7584,0.0519,0.1278
Zn,0.7870,-0.0262,-0.1038,-0.3767,-0.1716



Varimax-rotated PCA loadings (first 5 PCs)


,PC1,PC2,PC3,PC4,PC5
Co,0.4950,-0.0571,0.1132,0.3893,0.6739
Al,0.2927,0.0691,0.0795,0.5752,0.6510
Ni,0.8753,-0.1194,0.0840,0.2805,-0.0022
Mn,0.3680,-0.1008,0.3749,0.7555,0.0395
Fe,0.3952,-0.1816,0.0869,0.4314,0.2992
Cr,0.8098,-0.1471,0.2660,0.2910,0.2968
Cu,0.8031,-0.0883,0.2292,0.2130,0.3083
Hg,0.1720,-0.0719,0.0558,-0.0210,0.8904
Pb,0.3085,-0.0388,0.8227,0.2362,-0.2242
Zn,0.8202,-0.0879,0.1257,0.0168,0.3251



Original PCA scores from the fit stage


,StationID,PC1,PC2,PC3,PC4,PC5
0,003ABC,-0.9420,0.4268,1.5492,1.0301,1.9349
1,004ABC,-3.4659,-0.0834,-1.6081,-0.2754,-0.0296
2,005ABC,-2.9909,-0.4385,1.0710,-0.5193,2.0940
3,007ABC,3.5053,1.9372,0.5261,-0.0271,2.3087
4,008A,-1.9699,-0.1024,1.1764,-0.3319,2.3487
...,...,...,...,...,...,...
305,S99,-2.5076,-0.6398,0.1162,-0.7184,-0.4045
306,UBC1,-0.6351,-0.6503,-0.7196,1.6332,0.1219
307,UCC1,-0.8381,0.1675,-0.1779,1.1082,-0.2192
308,UCE1,-0.5970,0.4528,0.5245,1.1038,-0.1562
